# 5.2.1 Model 2: Ridge Regression

Regularized Linear Regression model. Trained on `X_train_scaled.csv` (log1p-transformed numeric features, per Section 3.11.2) since Ridge Regression is scale-sensitive. L2 regularization shrinks coefficients toward zero to reduce variance and improve generalization; the regularization strength (`alpha`) is tuned below rather than fixed.

In [1]:
import pandas as pd
import numpy as np
import sys, os
from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold
from sklearn.metrics import make_scorer
from skopt import BayesSearchCV
from skopt.space import Real
sys.path.append(os.path.dirname(os.path.abspath('__file__')))
from model_utils import evaluate_model, cross_validate_model

MODELLING_DIR = os.path.join("..", "data", "modelling")

X_train = pd.read_csv(os.path.join(MODELLING_DIR, "X_train_scaled.csv"))
X_test = pd.read_csv(os.path.join(MODELLING_DIR, "X_test_scaled.csv"))
y_train = pd.read_csv(os.path.join(MODELLING_DIR, "y_train.csv"))["price"]
y_test = pd.read_csv(os.path.join(MODELLING_DIR, "y_test.csv"))["price"]

print(f"X_train: {X_train.shape} | X_test: {X_test.shape}")

X_train: (3004, 51) | X_test: (751, 51)


## Hyperparameter Tuning
Tuned with `BayesSearchCV` (log-uniform search over `alpha`, `n_iter=50`, 5-fold CV on `X_train`). Since `y_train` is `log(price)` (Section 3.6), a plain RMSE scorer would rank by log-scale error - not guaranteed to match RM-scale RMSE, this project's primary metric (Section 1.8). A custom scorer converts predictions back to RM before scoring, same reasoning as Random Forest/KNN's tuning.

In [2]:
def rm_scale_rmse(y_true_log, y_pred_log):
    y_true_rm = np.exp(y_true_log)
    y_pred_rm = np.exp(y_pred_log)
    return np.sqrt(np.mean((y_true_rm - y_pred_rm) ** 2))

rm_rmse_scorer = make_scorer(rm_scale_rmse, greater_is_better=False)
cv_splitter = KFold(n_splits=5, shuffle=True, random_state=42)

search_spaces = {
    "alpha": Real(1e-3, 1e3, prior="log-uniform"),
}

bayes_search = BayesSearchCV(
    Ridge(),
    search_spaces,
    n_iter=50,
    scoring=rm_rmse_scorer,
    cv=cv_splitter,
    n_jobs=-1,
    random_state=42,
)
bayes_search.fit(X_train, y_train)

print("Best params:", bayes_search.best_params_)
print(f"Best CV score (RM-scale RMSE): RM {-bayes_search.best_score_:,.0f}")

Best params: OrderedDict({'alpha': 0.001})
Best CV score (RM-scale RMSE): RM 171,145


## 5-fold Cross-Validation
Run on `X_train` only, using `bayes_search.best_estimator_`. `X_test` is not touched here.

In [3]:
cv_results = cross_validate_model(
    bayes_search.best_estimator_,
    X_train, y_train, n_splits=5,
)

5-fold CV (mean +/- std):
  RMSE:  RM 171,145 +/- 27,001  (48.7% of median price)
  MAE:   RM 88,629 +/- 5,166
  MAPE:  21.8% +/- 1.1%
  R2:    0.7215 +/- 0.0488


## Train Final Model
`BayesSearchCV`'s `refit=True` already retrained the best config on the full `X_train` (`bayes_search.best_estimator_`), reused directly here. `X_test` is touched only once, below.

In [4]:
final_params = bayes_search.best_params_
ridge_model = bayes_search.best_estimator_

print("Final params:", final_params)
print("Model ready (reused from BayesSearchCV's refit=True).")

Final params: OrderedDict({'alpha': 0.001})
Model ready (reused from BayesSearchCV's refit=True).


## Evaluate
Metrics computed on both train and test sets for Section 6.2's overfitting/underfitting analysis.

In [5]:
train_metrics = evaluate_model(
    ridge_model,
    X_train,
    y_train,
    label="Train"
)

print()

test_metrics = evaluate_model(
    ridge_model,
    X_test,
    y_test,
    label="Test"
)

Train RMSE:  RM 168,245  (48.1% of median price)
Train MAE:   RM 86,459
Train MAPE:  21.2%
Train R2:    0.7366

Test RMSE:  RM 194,977  (54.2% of median price)
Test MAE:   RM 92,466
Test MAPE:  19.8%
Test R2:    0.6546


## Coefficients
Ridge coefficients are shrunk relative to ordinary Linear Regression, but their signs should broadly agree with the correlation directions observed in EDA (e.g. Property Size positive, Property Age negative) - a basic sanity check.

In [6]:
coef_table = pd.Series(
    ridge_model.coef_,
    index=X_train.columns
).sort_values(key=abs, ascending=False)

print("Top 15 coefficients by magnitude:")
print(coef_table.head(15))

Top 15 coefficients by magnitude:
Property Size                     0.842089
State_Perak                      -0.479708
Bedroom                          -0.433805
State_Negeri_Sembilan            -0.403820
PropertyType_Service_Residence    0.304372
Bathroom                          0.289583
State_Penang                      0.287016
State_Sabah                       0.283754
State_Melaka                     -0.272300
State_Pahang                      0.245374
PropertyType_Flat                -0.237358
State_Unknown                     0.217951
Parking Lot                       0.199368
State_Sarawak                     0.197708
State_Other                       0.172889
dtype: float64


## Save Trained Model
Saved for the Streamlit prototype (Section 8) to load directly, without retraining.

In [7]:
import joblib

MODEL_DIR = os.path.join("..", "models")
os.makedirs(MODEL_DIR, exist_ok=True)

model_path = os.path.join(MODEL_DIR, "ridge_model.pkl")

joblib.dump(ridge_model, model_path)

print(f"Model saved to {model_path}")

Model saved to ..\models\ridge_model.pkl
